# Mixture of Experts (MoE)

## Scaling Models Without Scaling Compute

A **dense** transformer scales both model quality *and* FLOPs together —
doubling parameters doubles compute per token.

**Sparse MoE** breaks this coupling:
- Replace each FFN block with $N$ expert FFNs
- A **routing / gating network** selects the top-$k$ experts for each token
- Each token uses only $k/N$ of total parameters → same compute, $N/k$× more capacity

```
Token x                       ┌─────────┐
       ──▶  Gate(x) ──▶ top-k ──▶ Expert_i │──▶  weighted sum ──▶ output
                               └─────────┘
                               ┌─────────┐
                          top-k──▶ Expert_j │──▶ /
                               └─────────┘
```

### Where MoE is Used
| Model | Experts | Top-k | Active params / token |
|---|---|---|---|
| Switch Transformer | 128-2048 | 1 | ~1 % |
| Mixtral 8×7B | 8 | 2 | 13 B (of 47 B) |
| DeepSeek-V3 | 256 | 8 | 37 B (of 671 B) |
| GPT-4 (rumoured) | ~8–16 | 2 | ~1/8 total |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")


## Implementing the MoE Layer

The core components are:
1. **Expert FFNs** — standard feed-forward networks (each independent)
2. **Gating network** — a linear layer that scores each expert for a given token
3. **Top-k selection** — keep only the $k$ highest-scoring experts and normalise their weights


In [ ]:
class ExpertFFN(nn.Module):
    """A single expert: two-layer feed-forward network with GELU activation."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MoELayer(nn.Module):
    """
    Sparse Mixture-of-Experts feed-forward layer.

    For each token:
      1. Compute gate scores for all N experts
      2. Select top-k experts
      3. Route token through those k experts
      4. Return weighted sum of expert outputs
    """
    def __init__(self, d_model: int, d_ff: int, num_experts: int = 8, top_k: int = 2):
        super().__init__()
        assert top_k <= num_experts
        self.num_experts = num_experts
        self.top_k       = top_k
        self.experts = nn.ModuleList([ExpertFFN(d_model, d_ff) for _ in range(num_experts)])
        self.gate    = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x: torch.Tensor):
        """
        x: (batch, seq, d_model)
        returns: (batch, seq, d_model), router_probs (batch*seq, num_experts)
        """
        B, T, D = x.shape
        x_flat = x.reshape(B * T, D)                       # (B*T, D)

        gate_logits = self.gate(x_flat)                    # (B*T, E)
        gate_probs  = torch.softmax(gate_logits, dim=-1)   # (B*T, E)

        # Select top-k experts per token
        topk_vals, topk_idx = torch.topk(gate_probs, self.top_k, dim=-1)  # (B*T, k)
        topk_vals = topk_vals / topk_vals.sum(dim=-1, keepdim=True)        # re-normalise

        output = torch.zeros_like(x_flat)

        # Route tokens to their assigned experts
        for k in range(self.top_k):
            expert_ids = topk_idx[:, k]          # which expert handles each token
            weights    = topk_vals[:, k]         # gate weight for that expert

            for e_idx in range(self.num_experts):
                token_mask = (expert_ids == e_idx)
                if token_mask.any():
                    expert_out = self.experts[e_idx](x_flat[token_mask])         # (n, D)
                    output[token_mask] += weights[token_mask].unsqueeze(-1) * expert_out

        return output.reshape(B, T, D), gate_probs


# ── Quick forward pass ────────────────────────────────────────────────────────
d_model, d_ff = 64, 256
moe = MoELayer(d_model=d_model, d_ff=d_ff, num_experts=8, top_k=2).to(device)
dense_ffn = ExpertFFN(d_model=d_model, d_ff=d_ff).to(device)

x = torch.randn(2, 16, d_model, device=device)   # batch=2, seq=16
out, router_probs = moe(x)
print(f"MoE input shape : {x.shape}")
print(f"MoE output shape: {out.shape}")
print(f"Router probs    : {router_probs.shape}  (B*T={2*16}, num_experts=8)")


In [ ]:
# ── Analyse routing distribution ─────────────────────────────────────────────
# Run a larger batch and see how tokens are distributed across experts

batch_x  = torch.randn(32, 64, d_model, device=device)   # 2048 tokens
_, probs = moe(batch_x)                                   # (2048, 8)

# Count which experts are actually selected (top-2)
topk_idx = torch.topk(probs, 2, dim=-1).indices.flatten()  # (2048*2,)
counts   = torch.bincount(topk_idx, minlength=8).float().cpu().numpy()
ideal    = len(topk_idx) / moe.num_experts

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: expert selection counts
axes[0].bar(range(8), counts, color="steelblue")
axes[0].axhline(ideal, color="red", ls="--", label=f"ideal = {ideal:.0f}")
axes[0].set(title="Expert selection counts (2048 tokens, top-2)",
            xlabel="Expert index", ylabel="Selections")
axes[0].legend()

# Right: mean gate probability per expert
mean_gate = probs.mean(dim=0).detach().cpu().numpy()
axes[1].bar(range(8), mean_gate, color="coral")
axes[1].axhline(1/8, color="red", ls="--", label="Uniform = 1/8")
axes[1].set(title="Mean gate probability per expert",
            xlabel="Expert index", ylabel="Mean gate weight")
axes[1].legend()

plt.suptitle("MoE Routing Analysis")
plt.tight_layout(); plt.show()

# Load imbalance metric
imbalance = counts.std() / counts.mean()
print(f"Routing imbalance (std/mean): {imbalance:.3f}  (0 = perfectly balanced)")


## Load Balancing Loss

Without regularisation, the gating network collapses — a few experts receive
most tokens (*expert collapse*).  The **auxiliary load-balancing loss** from
the Switch Transformer penalises imbalanced routing:

$$\mathcal{L}_\text{aux} = \alpha \cdot N \sum_{i=1}^{N} f_i \cdot P_i$$

where:
- $f_i$ = fraction of tokens *dispatched* to expert $i$
- $P_i$ = average gate probability assigned to expert $i$
- $\alpha$ = small coefficient (e.g. 0.01)
- $N$ = number of experts

Minimising $\mathcal{L}_\text{aux}$ encourages tokens to distribute uniformly.


In [ ]:
def load_balancing_loss(gate_probs: torch.Tensor, top_k_indices: torch.Tensor,
                        num_experts: int, alpha: float = 0.01) -> torch.Tensor:
    """
    Switch Transformer auxiliary load-balancing loss.
    gate_probs  : (B*T, E) — softmax gate probabilities
    top_k_indices: (B*T, k) — selected expert indices
    """
    T = gate_probs.shape[0]

    # f_i: fraction of tokens dispatched to expert i (uses one-hot dispatch)
    dispatch = torch.zeros(T, num_experts, device=gate_probs.device)
    for k in range(top_k_indices.shape[1]):
        dispatch.scatter_(1, top_k_indices[:, k:k+1], 1.0)
    f = dispatch.mean(dim=0)           # (E,)  fraction dispatched

    # P_i: mean gate probability for expert i
    P = gate_probs.mean(dim=0)         # (E,)

    return alpha * num_experts * (f * P).sum()


# ── Compare routing with and without load-balancing regularisation ─────────────
class MoELayerBalanced(MoELayer):
    """MoE layer that also returns the load-balancing loss."""
    def forward(self, x):
        B, T, D = x.shape
        x_flat  = x.reshape(B * T, D)
        gate_logits = self.gate(x_flat)
        gate_probs  = torch.softmax(gate_logits, dim=-1)
        topk_vals, topk_idx = torch.topk(gate_probs, self.top_k, dim=-1)
        topk_vals = topk_vals / topk_vals.sum(dim=-1, keepdim=True)

        output = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            for e in range(self.num_experts):
                mask = (topk_idx[:, k] == e)
                if mask.any():
                    output[mask] += topk_vals[mask, k:k+1] * self.experts[e](x_flat[mask])

        aux_loss = load_balancing_loss(gate_probs, topk_idx, self.num_experts)
        return output.reshape(B, T, D), gate_probs, aux_loss


moe_balanced = MoELayerBalanced(d_model=d_model, d_ff=d_ff, num_experts=8, top_k=2)
x_test = torch.randn(4, 32, d_model)
out, probs, aux = moe_balanced(x_test)
print(f"Output         : {out.shape}")
print(f"Aux loss       : {aux.item():.6f}")

# In the training loop you would do:
#   task_loss = criterion(output, targets)
#   total_loss = task_loss + aux_loss
#   total_loss.backward()
print("\nIn training: total_loss = task_loss + aux_loss")


In [ ]:
# ── Build a small MoE transformer block ───────────────────────────────────────

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        return self.norm(x + attn_out)

class MoETransformerBlock(nn.Module):
    """Transformer block where the FFN sub-layer is replaced by an MoE layer."""
    def __init__(self, d_model, num_heads, d_ff, num_experts=8, top_k=2):
        super().__init__()
        self.attn    = MultiHeadSelfAttention(d_model, num_heads)
        self.moe     = MoELayerBalanced(d_model, d_ff, num_experts, top_k)
        self.norm    = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = self.attn(x)
        moe_out, router_probs, aux_loss = self.moe(x)
        x = self.norm(x + self.dropout(moe_out))
        return x, router_probs, aux_loss

class DenseTransformerBlock(nn.Module):
    """Standard dense transformer block for comparison."""
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.ffn  = ExpertFFN(d_model, d_ff)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.attn(x)
        return self.norm(x + self.ffn(x))

# ── Parameter and FLOP comparison ─────────────────────────────────────────────
d_model, num_heads, d_ff = 128, 4, 512
N_experts, top_k = 8, 2

moe_block   = MoETransformerBlock(d_model, num_heads, d_ff, N_experts, top_k)
dense_block = DenseTransformerBlock(d_model, num_heads, d_ff)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

moe_params   = count_params(moe_block)
dense_params = count_params(dense_block)

# Active parameter count per token
attn_params = count_params(moe_block.attn)
expert_params = count_params(moe_block.moe.experts[0])
moe_active_params = attn_params + (top_k * expert_params)

dense_ffn_flops = 2 * (2 * d_model * d_ff)
moe_ffn_flops   = top_k * (2 * (2 * d_model * d_ff))

print(f"{'':22s} {'Dense Block':>14s} {'MoE (8 exp, top-2)':>22s}")
print(f"{'Total Parameters':22s} {dense_params:>14,} {moe_params:>22,}")
print(f"{'Active Params / Token':22s} {dense_params:>14,} {moe_active_params:>22,}")
print(f"{'FFN FLOPs / Token':22s} {dense_ffn_flops:>14,} {moe_ffn_flops:>22,}")
print(f"\nCapacity Gain: MoE provides {moe_params/dense_params:.1f}× total model capacity,")
print(f"while activating only {moe_active_params/moe_params*100:.1f}% of parameters per forward pass.")

# Forward pass verification
x = torch.randn(2, 16, d_model)
moe_out, rp, aux = moe_block(x)
dense_out        = dense_block(x)
print(f"\nMoE block output shape  : {moe_out.shape}  (aux_loss={aux.item():.6f})")
print(f"Dense block output shape: {dense_out.shape}")


---
## Modern MoE Innovations: Shared Experts & DeepSeek-V3

Recent state-of-the-art architectures (**DeepSeek-V2/V3**, **Qwen2.5-MoE**) address two major limitations of classical Switch / Mixtral MoE:

1. **Knowledge Fragmentation & Expert Redundancy**: When every token is routed only to specialized experts, common foundational patterns (punctuation, grammar, basic reasoning) get redundantly replicated across all experts.
   - **Solution**: **Dedicated Shared Expert(s)** ($N_{\text{shared}} \ge 1$) that are **always active for every token**, capturing common knowledge.
2. **Fine-Grained Expert Routing**: Instead of 8 large experts choosing top-2, DeepSeek-V3 uses **256 smaller experts** choosing **top-8**, enabling vastly more expressive combinations.

$$y = \text{FFN}_{\text{shared}}(x) + \sum_{i \in \text{top-}k} g_i \cdot \text{FFN}_{\text{routed}, i}(x)$$


In [ ]:
# DeepSeek / Qwen style Shared + Routed MoE Architecture
class SharedAndRoutedMoELayer(nn.Module):
    """
    MoE layer with 1 dedicated shared expert + N routed experts.
    As used in DeepSeek-V2/V3 and Qwen2.5-MoE.
    """
    def __init__(self, d_model: int, d_ff: int, num_routed_experts: int = 8, top_k: int = 2):
        super().__init__()
        self.num_routed = num_routed_experts
        self.top_k = top_k
        
        # 1. Dedicated shared expert (always executed)
        self.shared_expert = ExpertFFN(d_model, d_ff)
        
        # 2. Sparse routed experts
        self.routed_experts = nn.ModuleList([ExpertFFN(d_model, d_ff) for _ in range(num_routed_experts)])
        self.gate = nn.Linear(d_model, num_routed_experts, bias=False)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, T, D = x.shape
        x_flat = x.reshape(B * T, D)
        
        # A. Shared expert output (all tokens)
        shared_out = self.shared_expert(x_flat)
        
        # B. Routed experts with top-k gating
        gate_logits = self.gate(x_flat)
        gate_probs = torch.softmax(gate_logits, dim=-1)
        topk_vals, topk_idx = torch.topk(gate_probs, self.top_k, dim=-1)
        topk_vals = topk_vals / topk_vals.sum(dim=-1, keepdim=True)  # normalize weights
        
        routed_out = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            e_indices = topk_idx[:, k]
            weights = topk_vals[:, k:k+1]
            for e in range(self.num_routed):
                mask = (e_indices == e)
                if mask.any():
                    routed_out[mask] += weights[mask] * self.routed_experts[e](x_flat[mask])
        
        # Total output = shared + routed
        total_out = (shared_out + routed_out).reshape(B, T, D)
        return total_out, gate_probs

# Test DeepSeek-style MoE forward pass
deepseek_moe = SharedAndRoutedMoELayer(d_model=64, d_ff=256, num_routed_experts=8, top_k=2)
x_tokens = torch.randn(2, 16, 64)
out_tokens, g_probs = deepseek_moe(x_tokens)

print(f'Input shape : {x_tokens.shape}')
print(f'Output shape: {out_tokens.shape}')


## Summary

| Concept | Detail |
|---|---|
| **Sparse activation** | Only top-$k$ experts activate per token → FLOPs ≪ total params |
| **Expert collapse** | Without load-balancing loss, a few experts dominate |
| **Capacity** | Each expert has a *capacity buffer* — tokens exceeding capacity are dropped |
| **Communication** | In distributed settings, all-to-all routing across GPUs is the bottleneck |
| **DeepSeek-V3** | 671 B total, 37 B active; uses 256 experts, top-8, with fine-grained expert routing |
| **Mixtral 8×7B** | 8 experts, top-2; widely used open-source MoE |
